# Malayalam tokenizer — Kaggle run (gated rounds)

**Notebook settings:** Accelerator = **GPU T4 x2** · Internet = **On** · (private repo: add Kaggle Secret `GITHUB_TOKEN`)

Work in rounds. Set `ROUND` in the first code cell, then *Run All*: every cell for rounds ≤ `ROUND` runs, later ones are skipped.
Do not move to the next round until the gates of the current one print **PASS**.

| ROUND | what runs | gate (must pass before the next round) | time |
|---|---|---|---|
| 1 | get code, install, `selftest`, tests | 2 GPUs visible, selftest green, tests green | ~5-8 min |
| 2 | dry run on a small real sample | all datasets download, baselines load, both GPUs used, report written | ~10-20 min (estimate) |
| 3 | full selection (`FULL_STAGES`) | report exists; look at it before round 4 | hours |
| 4 | variation rounds you choose in `RUN_VARIATIONS` | compare snapshots; keep the config `name` to reuse cached work | per variation |
| 5 | freeze checklist + `freeze.json` + results.zip | all freeze gates pass → tag the commit, go to the DGX | minutes |

Every round only changes **config or plugin files** — never the engine. Edit on GitHub, `git push`, then re-run the *Get code* cell (it pulls).

In [ ]:
# ================= SETTINGS (edit here) =================
ROUND = 1                  # 1 setup+tests | 2 dry run | 3 full selection | 4 variations | 5 freeze + package
REPO   = "https://github.com/<you>/<repo>.git"     # <- your repository
BRANCH = "main"
COMMIT = ""                # pin an exact commit for recorded runs (empty = latest of BRANCH)
MIN_GPUS = 2               # Kaggle 'GPU T4 x2'. Set 1 if you only got one GPU, 0 to test on CPU.
RUN_TESTS = True           # False only if the tests already passed on this exact commit
DRYRUN_CFG = "configs/kaggle_dryrun.yaml"
FULL_CFG   = "configs/kaggle_small.yaml"
FULL_STAGES = ["intrinsic", "proxy"]   # round 3. "intrinsic" is CPU-only: run it with Accelerator=None to save GPU quota,
                                       # then run ["proxy"] in a GPU session (attach this notebook's output as input; restore cell below)
RUN_VARIATIONS = []        # round 4: e.g. ["seeds"], later ["vocab", "zwnj"]  (defined in the round-4 cell)
FREEZE_RUN = ""            # round 5: run name to freeze (empty = the name in FULL_CFG)

# ---- checkpoints -> Hugging Face (every finished unit of work is pushed; a fresh session restores it) ----
SYNC = True                # False = keep everything only in /kaggle/working (lost if the session dies)
HF_REPO = "Akzhai/mltok-malayalam-runs"   # a NEW private model repo (created if missing). Do NOT reuse an existing model repo.
HF_SECRET_NAMES = ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HF_WRITE_TOKEN"]   # Kaggle secret names to try
SYNC_DATA = False          # True also pushes the cleaned data shards (~GBs) so a restart needs no re-download/re-clean
CKPT_PUSH_S = 900          # mid-training proxy checkpoints are pushed at most this often (seconds)
STORE = None               # advanced: any `store` plugin spec, e.g. {"type": "local_dir", "path": "/kaggle/working/backup"}
DONE_ROUNDS = []           # rounds whose gates ALREADY passed in an earlier session, e.g. [1, 2]: their runs are skipped
                           # (get-code + install always re-run: every Kaggle session starts on a fresh machine)
# ========================================================

import glob, json, os, shlex, shutil, subprocess, sys, time

WORK = os.environ.get("MLTOK_WORK", "/kaggle/working")
SRC = f"{WORK}/mltok-pipeline"
GATES = {}
G1 = ["gpus visible", "selftest", "tests", "checkpoint store"]
G2 = ["dry: all datasets downloaded", "dry: report written"]
_SECRETS = []                                    # strings masked in all printed commands (tokens)


def need(r, title=""):
    ok = ROUND >= r
    if not ok:
        print(f"[skip] round {r} not enabled (ROUND={ROUND}) {title}")
    return ok


def sh(cmd, cwd=None, check=True):
    shown = cmd
    for s in _SECRETS:
        shown = shown.replace(s, "***")
    print(f"$ {shown}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        for s in _SECRETS:
            line = line.replace(s, "***")
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc:
        raise RuntimeError(f"command failed (exit {rc}): {shown}")
    return rc


def store_spec():
    return STORE or {"type": "hf_hub", "repo": HF_REPO, "private": True, "prefix": "mltok"}


def sync_spec():
    return {"enabled": bool(SYNC), "min_interval_s": 120, "ckpt_push_interval_s": CKPT_PUSH_S,
            "restore": True, "include_cache": True, "include_checkpoints": True, "include_data": bool(SYNC_DATA)}


_SYNC_CMDS = ("ingest", "clean", "run", "restore", "push", "store-check")


def mt(args, check=True):
    """Run `python -m mltok <args>` inside the repo. Config-driven commands get the checkpoint store appended."""
    if SYNC and args.split()[0] in _SYNC_CMDS and " -c " in f" {args} ":
        extra = " ".join(shlex.quote(f"{k}={json.dumps(v)}") for k, v in (("store", store_spec()), ("sync", sync_spec())))
        args = f"{args} {extra}" if " --set " in f" {args} " else f"{args} --set {extra}"
    return sh(f"{sys.executable} -m mltok {args}", cwd=SRC, check=check)


def trust(names, r):
    for n in names:
        GATES[n] = True
    print(f"[trusted] round {r} gates from an earlier session: {names}")


def gate(name, ok, detail=""):
    GATES[name] = bool(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {name} {detail}")
    return bool(ok)


def require(*names):
    bad = [n for n in names if not GATES.get(n)]
    if bad:
        raise SystemExit(f"Stop: these gates have not passed (fix them first): {bad}")


def gpu_count():
    r = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True)
    return len([ln for ln in r.stdout.splitlines() if ln.startswith("GPU")]) if r.returncode == 0 else 0


def load_cfg(rel):
    import yaml
    return yaml.safe_load(open(f"{SRC}/{rel}"))


def run_dir(cfg, name=None):
    return f"{cfg['paths']['runs']}/{name or cfg['name']}"


def show(path, limit=12000):
    print(open(path).read()[:limit] if os.path.exists(path) else f"(missing: {path})")


print("settings ok | work dir:", WORK)

## Round 1 — setup and tests

In [ ]:
# --- get the code from GitHub (re-run this cell after every `git push` to update) ---
if need(1, "(get code)"):
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")      # only needed for a private repo
    except Exception:
        pass
    url = REPO.replace("https://", f"https://{token}@", 1) if token and REPO.startswith("https://") else REPO
    if token:
        _SECRETS.append(token)
    if os.path.exists(f"{SRC}/.git"):
        sh(f"git remote set-url origin {shlex.quote(url)} && git fetch origin && git checkout {BRANCH} && git pull --ff-only", cwd=SRC)
    else:
        sh(f"git clone --branch {BRANCH} {shlex.quote(url)} {shlex.quote(SRC)}", cwd=WORK)
    sh(f"git remote set-url origin {shlex.quote(REPO)}", cwd=SRC)        # never keep the token in .git/config
    if COMMIT:
        sh(f"git checkout {COMMIT}", cwd=SRC)
    sh("git log -1 --oneline", cwd=SRC)
    COMMIT_HASH = subprocess.check_output(["git", "-C", SRC, "rev-parse", "HEAD"], text=True).strip()
    assert os.path.exists(f"{SRC}/mltok/__init__.py"), "repository does not contain the pipeline"

In [ ]:
# --- offline alternative: pipeline uploaded as a Kaggle Dataset (zip) instead of GitHub ---
# hits = glob.glob('/kaggle/input/**/mltok/__init__.py', recursive=True); assert hits, "attach the dataset first"
# shutil.copytree(hits[0].rsplit('/mltok/__init__.py', 1)[0], SRC, dirs_exist_ok=True)

In [ ]:
if need(1, "(install)"):
    sh("pip install -q -r requirements.txt datasets transformers", cwd=SRC)
    mt("sysinfo")

In [ ]:
# --- Hugging Face token from Kaggle Secrets (value is never printed; it is masked in every log line) ---
HF_FOUND = None
if need(1, "(hf token)") and SYNC and STORE is None:
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for nm in HF_SECRET_NAMES:
            try:
                val = client.get_secret(nm)
            except Exception:
                continue
            if val:
                os.environ["HF_TOKEN"] = val          # inherited by the proxy worker processes
                _SECRETS.append(val)
                HF_FOUND = nm
                break
    except Exception:
        pass
    if "HF_TOKEN" in os.environ and not HF_FOUND:
        HF_FOUND = "HF_TOKEN (environment)"
        _SECRETS.append(os.environ["HF_TOKEN"])
    print("Hugging Face token:", f"found in secret/env '{HF_FOUND}'" if HF_FOUND else
          "NOT FOUND - add a Kaggle Secret (Add-ons > Secrets) with write access and attach it to this notebook")

In [ ]:
# --- GATE 1 ---
if need(1, "(gate 1)"):
    if 1 in DONE_ROUNDS:
        trust(G1, 1)
    else:
        n = gpu_count()
        gate("gpus visible", n >= MIN_GPUS, f"({n} found, need >= {MIN_GPUS}; Accelerator must be 'GPU T4 x2')")
        gate("selftest", mt("selftest", check=False) == 0, "(GPU training on each GPU + both at once)")
        if RUN_TESTS:
            gate("tests", sh(f"{sys.executable} -m pytest tests -q -x", cwd=SRC, check=False) == 0, "(contracts, plug-in swap, trainer, pipeline)")
        else:
            gate("tests", True, "(SKIPPED by RUN_TESTS=False)")
        if SYNC:
            gate("checkpoint store", mt(f"store-check -c {DRYRUN_CFG}", check=False) == 0,
                 f"(push -> pull round trip to {store_spec().get('repo') or store_spec().get('type')})")
        else:
            gate("checkpoint store", True, "(SKIPPED: SYNC=False - nothing is saved outside this session!)")
        print("\nGATE 1:", "ALL PASSED -> set ROUND = 2" if all(GATES[g] for g in G1) else "FIX THE FAILURES ABOVE FIRST")

## Round 2 — dry run on a small real sample
Proves dataset ids, Hub baselines, both GPUs and the report before the long run. Fix a failing dataset id in `configs/*.yaml` on GitHub, push, re-run *Get code*, re-run this cell.

In [ ]:
if need(2, "(dry run)"):
    require(*G1)
    if 2 in DONE_ROUNDS:
        trust(G2, 2)
    else:
        dcfg = load_cfg(DRYRUN_CFG)
        mt(f"ingest -c {DRYRUN_CFG}")
        mt(f"clean  -c {DRYRUN_CFG}")
        mt(f"run    -c {DRYRUN_CFG}")
        root = run_dir(dcfg)
        ids = [s.get("id", s["type"]) for s in dcfg["sources"]]
        missing = [i for i in ids if not os.path.exists(f"{dcfg['paths']['raw']}/{i}.jsonl.gz")]
        gate("dry: all datasets downloaded", not missing, f"(missing: {missing})")
        gate("dry: report written", os.path.exists(f"{root}/report.md"))
        if dcfg.get("baselines"):
            got = glob.glob(f"{root}/cache/baseline_*.json")
            gate("dry: Hub baselines loaded", len(got) == len(dcfg["baselines"]), f"({len(got)}/{len(dcfg['baselines'])}; gated/unavailable ones are skipped)")
        if gpu_count() >= 2:
            gate("dry: both GPUs used by proxy workers", all(os.path.exists(f"{root}/proxy_worker_gpu{i}.log") for i in range(2)))
        show(f"{root}/report.md")
        print("\nGATE 2:", "ALL PASSED -> set ROUND = 3" if all(GATES[g] for g in GATES if g.startswith("dry:")) else "FIX THE FAILURES ABOVE FIRST (send me report.md + the log if unsure)")

## Round 3 — full selection

In [ ]:
# --- restore finished work after a session restart: from the checkpoint store, else from an attached notebook output ---
if need(3, "(restore)"):
    fcfg = load_cfg(FULL_CFG)
    if SYNC:
        mt(f"restore -c {FULL_CFG} --what {'all' if SYNC_DATA else 'run'}")
    for variants in glob.glob(f"/kaggle/input/**/{fcfg['name']}/variants", recursive=True):
        dst = run_dir(fcfg)
        shutil.copytree(os.path.dirname(variants), dst, dirs_exist_ok=True)
        print("restored run dir from", os.path.dirname(variants))
    for stats in glob.glob("/kaggle/input/**/clean/clean_stats.json", recursive=True):
        shutil.copytree(os.path.dirname(stats), fcfg["paths"]["clean"], dirs_exist_ok=True)
        print("restored cleaned data from", os.path.dirname(stats))

In [ ]:
if need(3, "(data)"):
    require(*G1, *G2)
    fcfg = load_cfg(FULL_CFG)
    if os.path.exists(f"{fcfg['paths']['clean']}/clean_stats.json"):
        print("cleaned data already present - skipping ingest/clean")
    else:
        mt(f"ingest -c {FULL_CFG}")
        mt(f"clean  -c {FULL_CFG}")
    show(f"{fcfg['paths']['clean']}/clean_stats.json")

In [ ]:
if need(3, "(selection)"):
    fcfg = load_cfg(FULL_CFG)
    for stage in FULL_STAGES:                       # "intrinsic" (CPU) then "proxy" (both GPUs)
        print(f"\n===== round 3: stage {stage} =====")
        mt(f"run -c {FULL_CFG} --stage {stage}")
    show(f"{run_dir(fcfg)}/report.md")

## Round 4 — variation rounds
Pick tags in `RUN_VARIATIONS` (first cell) and re-run this cell. Each variation edits **config only** (via `--set`); with the same run `name`, finished work is reused.
A short report/CSV snapshot of every variation is kept in `snapshots/<tag>/`. Suggested order: `seeds` → `vocab` → `zwnj` → `algos` → `bigproxy` → `datamix`.
Treat BPB differences smaller than the seed spread as ties.

In [ ]:
FULL_NAME = load_cfg(FULL_CFG)["name"] if os.path.exists(f"{SRC}/{FULL_CFG}") else "kaggle_v2"

VARIATIONS = {
    "seeds":    dict(note="noise control: top-2 with 3 proxy seeds -> mean +/- std",
                     sets={"proxy.seeds": [0, 1, 2], "proxy.top_k": 2}),
    "vocab":    dict(note="vocabulary sweep for the winning pipeline",
                     sets={"proxy.vocab_sweep": [16000, 24000, 32000, 48000]}),
    "zwnj":     dict(note="normalizer / ZWNJ-ZWJ policy",
                     sets={"search.factors": {"normalizer": ["none", "nfc", "ml", {"type": "ml", "zw": "drop"}]}}),
    "algos":    dict(note="algorithm family",
                     sets={"search.factors": {"algorithm": ["mlbpe", "mlbpe_plain", "hf_bpe", "spm_unigram", "spm_bpe"]}}),
    "bigproxy": dict(note="4x more proxy training text (new run name keeps the 120 MB results)",
                     sets={"proxy.train_mb": 400, "name": FULL_NAME + "_bigproxy"}),
    "datamix":  dict(note="data mix check: web only (own data folders + run name, nothing of the main run is touched)",
                     sets={"name": FULL_NAME + "_webonly",
                           "paths.raw": f"{WORK}/data/raw_webonly", "paths.clean": f"{WORK}/data/clean_webonly",
                           "sources": [{"type": "hf_stream", "id": "fineweb2", "hf": "HuggingFaceFW/fineweb-2",
                                        "name": "mal_Mlym", "split": "train", "text": "text", "max_docs": 250000}]}),
    # add your own: "mytag": dict(note="...", sets={"dotted.config.key": value, ...})
}


def variation(tag, note, sets, stage="all"):
    cfg = load_cfg(FULL_CFG)
    name = sets.get("name", cfg["name"])
    args = " ".join(shlex.quote(f"{k}={json.dumps(v)}") for k, v in sets.items())
    print(f"\n===== variation '{tag}': {note} =====")
    if "sources" in sets:                      # different data: must NOT share (and overwrite) the main run's folders
        assert "paths.raw" in sets and "paths.clean" in sets and "name" in sets,             "a variation that changes `sources` must also set name, paths.raw and paths.clean"
        os.makedirs(sets["paths.raw"], exist_ok=True)
        for src in sets["sources"]:             # reuse already-downloaded raw files instead of downloading again
            f = f"{cfg['paths']['raw']}/{src.get('id', src['type'])}.jsonl.gz"
            if os.path.exists(f) and not os.path.exists(f"{sets['paths.raw']}/{os.path.basename(f)}"):
                shutil.copy(f, sets["paths.raw"])
        mt(f"ingest -c {FULL_CFG} --set {args}")
        mt(f"clean  -c {FULL_CFG} --set {args}")
    mt(f"run -c {FULL_CFG} --stage {stage} --set {args}")
    root = run_dir(cfg, name)
    snap = f"{WORK}/snapshots/{tag}"
    os.makedirs(snap, exist_ok=True)
    for f in ("report.md", "results.csv"):
        if os.path.exists(f"{root}/{f}"):
            shutil.copy(f"{root}/{f}", snap)
    print(f"snapshot -> {snap}")
    show(f"{root}/report.md", 6000)


if need(4, "(variations)"):
    require(*G1)
    for tag in RUN_VARIATIONS:
        variation(tag, **VARIATIONS[tag])
    if not RUN_VARIATIONS:
        print("RUN_VARIATIONS is empty - choose from:", list(VARIATIONS))

## Round 5 — freeze checklist
Checks that the selected tokenizer is safe to hand to the DGX. All gates must pass; the freeze record (`freeze.json`) stores the commit, config, winner, plugin file hashes and data fingerprints.

In [ ]:
if need(5, "(freeze)"):
    require(*G1)
    fcfg = load_cfg(FULL_CFG)
    name = FREEZE_RUN or fcfg["name"]
    root = run_dir(fcfg, name)
    sel = json.load(open(f"{root}/best/SELECTION.json"))
    m, p = sel["metrics"], sel.get("proxy") or {}
    print("winner:", sel["winner"], "|", sel["basis"])

    gate("freeze: winner is lossless on eval docs and stress set",
         m.get("roundtrip_ok", 1) >= 0.999 and m.get("stress_roundtrip_ok", 1) >= 1.0 and m.get("unk_rate", 0) <= 1e-3,
         f"(roundtrip={m.get('roundtrip_ok')}, stress={m.get('stress_roundtrip_ok')}, unk={m.get('unk_rate')})")

    # the tokenizer must be usable by a training stack: a Hugging Face tokenizer.json (SentencePiece winners are exported automatically)
    has_json = os.path.exists(f"{root}/best/tokenizer.json")
    gate("freeze: best/ has tokenizer.json (Hugging Face format)", has_json,
         "" if has_json else "(older run? convert with: python -m mltok export-hf --spm <best>/spm.model --normalizer ml)")
    gate("freeze: verify passes on best/", has_json and mt(f"verify --tokenizer {root}/best", check=False) == 0)
    gate("freeze: seed noise measured (>= 2 seeds)", p.get("n_seeds", 1) >= 2, "(run variation 'seeds')")

    # guard against the data-mix bug of the first run (training set filled by the first source alone)
    cs = f"{fcfg['paths']['clean']}/clean_stats.json"
    if os.path.exists(cs):
        share = json.load(open(cs)).get("train_share", {})
        n_src = len(fcfg.get("sources", []))
        ok_mix = n_src <= 1 or (len(share) >= n_src and max(share.values() or [1]) <= 0.9)
        gate("freeze: training data uses every source (no source > 90%)", ok_mix, f"(train share: {share})")
    else:
        print("[note] clean_stats.json not found - training-data mix was not checked")

    import csv
    rows = list(csv.DictReader(open(f"{root}/results.csv")))
    def bpb(r):
        return float(r["proxy_bpb"]) if r.get("proxy_bpb") not in (None, "", "-") else None
    def is_elig(r):
        return str(r.get("eligible")).lower() == "true"
    others = sorted((bpb(r), r["name"]) for r in rows if r["name"] != sel["winner"] and bpb(r) is not None and is_elig(r))
    if others and p:
        gap, noise = others[0][0] - p["bpb"], p.get("bpb_std", 0.0)
        gate("freeze: winner beats the best OTHER eligible variant by more than seed noise", gap > noise,
             f"(gap {gap:.4f} BPB vs seed std {noise:.4f}; runner-up {others[0][1]})")
    else:
        gate("freeze: winner beats the best OTHER eligible variant by more than seed noise", False, "(no other eligible variant has a proxy result)")
    better_inelig = sorted((bpb(r), r["name"]) for r in rows if bpb(r) is not None and not is_elig(r) and p and bpb(r) < p["bpb"])
    if better_inelig:
        print(f"[note] ineligible variant {better_inelig[0][1]} scored better ({better_inelig[0][0]:.4f} vs {p['bpb']:.4f}) but is not lossless - "
              f"it cannot be chosen; a lossless version of it (byte fallback) may be worth testing")
    custom = sorted((bpb(r), r["name"]) for r in rows if r["name"].startswith("mlbpe") and bpb(r) is not None)
    if custom and p and not sel["winner"].startswith("mlbpe"):
        print(f"[note] our custom BPE did NOT win: best {custom[0][1]} = {custom[0][0]:.4f} BPB vs winner {p['bpb']:.4f} "
              f"(+{(custom[0][0] / p['bpb'] - 1) * 100:.1f}%). The structure metrics (mark_start / akshara_break) did not translate into lower BPB.")
    pub = [json.load(open(f))["fertility"] for f in glob.glob(f"{root}/cache/baseline_*.json") if "fertility" in json.load(open(f))]
    if pub:
        gate("freeze: fertility below every published tokenizer", m["fertility"] < min(pub),
             f"(ours {m['fertility']:.3f} vs best published {min(pub):.3f}; NOT like-for-like: ours is Malayalam-only and in-domain)")
    else:
        print("[note] no published-tokenizer baselines loaded - fertility vs published tokenizers was not checked")

    G5 = [g for g in GATES if g.startswith("freeze:")]
    commit = subprocess.check_output(["git", "-C", SRC, "rev-parse", "HEAD"], text=True).strip()
    freeze = {"commit": commit, "config": FULL_CFG, "run": name, "winner": sel["winner"], "gates": {g: GATES[g] for g in G5},
              "provenance": sel.get("provenance"), "metrics": m, "proxy": p}
    json.dump(freeze, open(f"{root}/freeze.json", "w"), indent=2, default=str)
    if SYNC:
        mt(f"push -c {FULL_CFG} --set {shlex.quote('name=' + json.dumps(name))}", check=False)
    print("\nGATE 5:", "ALL PASSED" if all(GATES[g] for g in G5) else "NOT READY - see FAIL lines")
    if all(GATES[g] for g in G5):
        print(f"\nNext: on your machine:  git tag v1-selected {commit} && git push origin v1-selected")
        print("Then on the DGX: git clone ... && git checkout v1-selected && scripts/run_dgx.sh <gpu_index>")

In [ ]:
# --- package results for download (Output tab) ---
if need(5, "(package)"):
    fcfg = load_cfg(FULL_CFG)
    name = FREEZE_RUN or fcfg["name"]
    root = run_dir(fcfg, name)
    stage = f"{WORK}/results_pkg"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)
    for f in ("report.md", "results.csv", "freeze.json", "timings.json"):
        if os.path.exists(f"{root}/{f}"):
            shutil.copy(f"{root}/{f}", stage)
    shutil.copytree(f"{root}/best", f"{stage}/best")
    if os.path.exists(f"{WORK}/snapshots"):
        shutil.copytree(f"{WORK}/snapshots", f"{stage}/snapshots")
    out = shutil.make_archive(f"{WORK}/results", "zip", stage)
    print("wrote", out, f"({os.path.getsize(out) / 1e6:.1f} MB) - download it from the notebook's Output tab")